## 03g_ipps_payment_rates — RCM Intelligence Platform
### Purpose
Creates Gold analytical tables that enrich IPPS payment rate data with
hospital performance metrics for strategic decision-making.

### What this does
1. Joins IPPS payment impacts with Hospital 360 Scorecard
2. Creates hospital-level payment rate table with revenue gap context
3. Aggregates payment trends by state
4. Identifies top-priority hospitals (high gap + payment volatility)

### Outputs
- rcm_platform.rcm_gold.ipps_hospital_payment_rates
- rcm_platform.rcm_gold.ipps_state_payment_summary
- rcm_platform.rcm_gold.dashboard_ipps_enriched_opportunities

### Notes
- Surfaces hospitals where declining Medicare rates compound revenue challenges
- Priority scoring weighs payment decreases more heavily than increases
- State summaries feed the State Intelligence page

| Field      | Value |
|------------|-------|
| Author     | Mayank Joshi |
| Layer      | Gold |
| Run order  | After 02g_process_ipps_final_rule |
| Depends on | 00_config, 00_utils, Silver IPPS, Hospital 360 Scorecard |

In [0]:
%run /Users/jmayank574@gmail.com/rcm-intelligence-platform/00_setup/00_config

In [0]:
%run /Users/jmayank574@gmail.com/rcm-intelligence-platform/00_setup/00_utils

In [0]:
# ============================================================
# BATCH METADATA
# ============================================================

import uuid
from datetime import datetime, timezone

BATCH_ID        = str(uuid.uuid4())
BATCH_DATE      = datetime.now(timezone.utc).strftime("%Y-%m-%d")
BATCH_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
NOTEBOOK_NAME   = "03g_ipps_payment_rates"

print(f"Batch ID        : {BATCH_ID}")
print(f"Batch date      : {BATCH_DATE}")
print(f"Batch timestamp : {BATCH_TIMESTAMP}")

In [0]:
# ============================================================
# TABLE 1: ipps_hospital_payment_rates
# Hospital-level IPPS data joined with Hospital 360 Scorecard
# ============================================================

from pyspark.sql.functions import col, when, lit, round as spark_round

print("=" * 60)
print("  CREATING ipps_hospital_payment_rates")
print("=" * 60)

# Read Silver IPPS data
df_ipps = spark.table(f"{SILVER_DB}.ipps_payment_impacts")

# Read Hospital 360 Scorecard
df_scorecard = spark.table(f"{GOLD_DB}.hospital_360_scorecard")

print(f"IPPS Silver rows        : {df_ipps.count():,}")
print(f"Hospital 360 Scorecard  : {df_scorecard.count():,}")

# Select and prepare IPPS columns
df_ipps_selected = df_ipps.select(
    col("provider_number"),
    col("hospital_name"),
    col("fy2023_operating_payment"),
    col("fy2024_operating_payment"),
    col("operating_payment_change_dollars"),
    col("operating_payment_change_pct"),
    col("wage_index_impact_category"),
    col("fy_2024_wage_index"),
    col("is_safety_net_hospital"),
    col("is_teaching_hospital"),
    col("dsh_percentage"),
    col("total_payment_fy2023"),
    col("total_payment_fy2024"),
    col("total_payment_change_dollars"),
    col("total_payment_change_pct")
)

# Left join to Hospital 360 Scorecard
df_joined = df_ipps_selected.join(
    df_scorecard.select(
        col("provider_id"),
        col("provider_state"),
        col("hospital_360_grade"),
        col("combined_revenue_gap_billions")
    ),
    df_ipps_selected.provider_number == df_scorecard.provider_id,
    "left"
).drop("provider_id")

# Add derived columns
df_hospital_rates = df_joined \
    .withColumn("payment_trend",
        when(col("operating_payment_change_pct") > 0, "Increasing")
        .when(col("operating_payment_change_pct") < 0, "Decreasing")
        .otherwise("Flat")
    ) \
    .withColumn("has_claims_data",
        when(col("hospital_360_grade").isNotNull(), 1).otherwise(0)
    ) \
    .withColumn("_gold_processed_at", lit(BATCH_TIMESTAMP).cast("timestamp")) \
    .withColumn("_gold_batch_id", lit(BATCH_ID))

# Write to Gold
target_table_1 = f"{GOLD_DB}.ipps_hospital_payment_rates"

df_hospital_rates.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(target_table_1)

print(f"\n✓ Created: {target_table_1}")
print(f"  Rows written: {spark.table(target_table_1).count():,}")

# JOIN COVERAGE CHECK
print("\n" + "=" * 60)
print("  JOIN COVERAGE ANALYSIS")
print("=" * 60)

spark.sql(f"""
    SELECT 
        COUNT(*) as total_ipps_hospitals,
        SUM(has_claims_data) as matched_to_scorecard,
        COUNT(*) - SUM(has_claims_data) as unmatched,
        ROUND(SUM(has_claims_data) * 100.0 / COUNT(*), 1) as match_rate_pct
    FROM {target_table_1}
""").show(truncate=False)

print("\nPayment trend breakdown:")
spark.sql(f"""
    SELECT
        payment_trend,
        COUNT(*) as hospital_count,
        SUM(has_claims_data) as with_claims_data
    FROM {target_table_1}
    GROUP BY payment_trend
    ORDER BY hospital_count DESC
""").show(truncate=False)

In [0]:
# ============================================================
# TABLE 2: ipps_state_payment_summary
# State-level aggregates for State Intelligence page
# ============================================================

from pyspark.sql.functions import count, avg, sum as spark_sum, round as spark_round

print("\n" + "=" * 60)
print("  CREATING ipps_state_payment_summary")
print("=" * 60)

# Aggregate by state
df_state_summary = spark.table(target_table_1) \
    .filter(col("provider_state").isNotNull()) \
    .groupBy("provider_state") \
    .agg(
        count("*").alias("hospital_count"),
        spark_sum("has_claims_data").alias("hospitals_with_claims"),
        spark_round(avg("operating_payment_change_pct"), 2).alias("avg_payment_change_pct"),
        spark_round(spark_sum("operating_payment_change_dollars"), 2).alias("total_payment_change_dollars"),
        spark_sum(when(col("payment_trend") == "Decreasing", 1).otherwise(0)).alias("hospitals_with_declining_payments"),
        spark_round(avg(when(col("has_claims_data") == 1, col("combined_revenue_gap_billions"))), 2).alias("avg_revenue_gap_billions")
    ) \
    .withColumn("_gold_processed_at", lit(BATCH_TIMESTAMP).cast("timestamp")) \
    .withColumn("_gold_batch_id", lit(BATCH_ID))

# Write to Gold
target_table_2 = f"{GOLD_DB}.ipps_state_payment_summary"

df_state_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(target_table_2)

print(f"\n✓ Created: {target_table_2}")
print(f"  Rows written: {spark.table(target_table_2).count():,}")

print("\nTop 5 states by declining payment impact:")
spark.sql(f"""
    SELECT
        provider_state,
        hospital_count,
        hospitals_with_declining_payments,
        avg_payment_change_pct,
        total_payment_change_dollars
    FROM {target_table_2}
    WHERE avg_payment_change_pct IS NOT NULL
    ORDER BY avg_payment_change_pct ASC
    LIMIT 5
""").show(truncate=False)

In [0]:
# ============================================================
# TABLE 3: dashboard_ipps_enriched_opportunities
# Top 20 hospitals with high gap + payment volatility
# ============================================================

from pyspark.sql.functions import abs as spark_abs

print("\n" + "=" * 60)
print("  CREATING dashboard_ipps_enriched_opportunities")
print("=" * 60)

# Filter to hospitals with claims data and calculate priority score
df_opportunities = spark.table(target_table_1) \
    .filter(col("combined_revenue_gap_billions").isNotNull()) \
    .withColumn("priority_score",
        when(col("operating_payment_change_pct") < 0,
            # Declining payments: 2x weight (divide by 50 instead of 100)
            col("combined_revenue_gap_billions") * (1 + spark_abs(col("operating_payment_change_pct")) / 50)
        ).when(col("operating_payment_change_pct") > 0,
            # Increasing payments: 1x weight
            col("combined_revenue_gap_billions") * (1 + col("operating_payment_change_pct") / 100)
        ).otherwise(
            # Flat payment: just use gap
            col("combined_revenue_gap_billions")
        )
    ) \
    .withColumn("_gold_processed_at", lit(BATCH_TIMESTAMP).cast("timestamp")) \
    .withColumn("_gold_batch_id", lit(BATCH_ID)) \
    .orderBy(col("priority_score").desc()) \
    .limit(20)

# Write to Gold
target_table_3 = f"{GOLD_DB}.dashboard_ipps_enriched_opportunities"

df_opportunities.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(target_table_3)

print(f"\n✓ Created: {target_table_3}")
print(f"  Rows written: {spark.table(target_table_3).count():,}")

print("\nTop 10 priority hospitals:")
spark.sql(f"""
    SELECT
        provider_number,
        hospital_name,
        provider_state,
        ROUND(combined_revenue_gap_billions, 2) as gap_billions,
        operating_payment_change_pct,
        payment_trend,
        ROUND(priority_score, 2) as priority_score
    FROM {target_table_3}
    ORDER BY priority_score DESC
    LIMIT 10
""").show(truncate=False)

In [0]:
# ============================================================
# VERIFICATION SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("  GOLD LAYER VERIFICATION")
print("=" * 60)

print("\n1. Hospital Payment Rates:")
spark.sql(f"""
    SELECT
        COUNT(*) as total_hospitals,
        SUM(has_claims_data) as with_scorecard_data,
        COUNT(DISTINCT provider_state) as states_covered
    FROM {GOLD_DB}.ipps_hospital_payment_rates
""").show(truncate=False)

print("\n2. State Payment Summary:")
spark.sql(f"""
    SELECT
        COUNT(*) as total_states,
        ROUND(AVG(hospital_count), 1) as avg_hospitals_per_state,
        ROUND(AVG(avg_payment_change_pct), 2) as overall_avg_payment_change_pct
    FROM {GOLD_DB}.ipps_state_payment_summary
""").show(truncate=False)

print("\n3. Enriched Opportunities:")
spark.sql(f"""
    SELECT
        COUNT(*) as opportunity_count,
        COUNT(DISTINCT provider_state) as states_represented,
        ROUND(AVG(combined_revenue_gap_billions), 2) as avg_gap_billions,
        ROUND(AVG(operating_payment_change_pct), 2) as avg_payment_change_pct
    FROM {GOLD_DB}.dashboard_ipps_enriched_opportunities
""").show(truncate=False)

print("\n" + "=" * 60)
print("  GOLD PROCESSING COMPLETE")
print("=" * 60)

In [0]:
# ============================================================
# AUDIT LOGGING
# ============================================================

# Count total rows across all three tables
total_rows_written = (
    spark.table(f"{GOLD_DB}.ipps_hospital_payment_rates").count() +
    spark.table(f"{GOLD_DB}.ipps_state_payment_summary").count() +
    spark.table(f"{GOLD_DB}.dashboard_ipps_enriched_opportunities").count()
)

log_pipeline_run(
    notebook_name    = NOTEBOOK_NAME,
    layer            = "gold",
    status           = "SUCCESS",
    rows_read        = spark.table(f"{SILVER_DB}.ipps_payment_impacts").count(),
    rows_written     = total_rows_written,
    rows_quarantined = 0,
    message          = f"Batch {BATCH_ID} — Created 3 Gold IPPS tables: hospital_payment_rates, state_payment_summary, enriched_opportunities"
)

print("\n✓ Pipeline run logged to audit table")